# R1000 Top30 Institutional Engine

Colab runbook for the current GitHub-based engine.

Flow:
1. Install dependencies
2. Mount Drive and load the project
3. Run collector
4. Run full pipeline
5. Run validation and inspect sleeve/rebalance outputs


In [ ]:
%pip install -q --upgrade pip
%pip install -q catboost pandas-market-calendars yfinance pyarrow openpyxl requests scikit-learn

import os
os.environ["ALPHA_VANTAGE_API_KEY"] = "JOUOW3UV8ZV23AOZ"

print("deps + env ready")


In [ ]:
from google.colab import drive
import os
import sys
import json
import importlib
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

drive.mount('/content/drive', force_remount=False)

BASE_DIR = '/content/drive/MyDrive/r1000_top30_institutional'
PROJECT_DIR = Path(BASE_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.chdir(PROJECT_DIR)

for name in ['r1000_top30_institutional', 'r1000_data_collector']:
    if name in sys.modules:
        del sys.modules[name]
importlib.invalidate_caches()

END_DATE = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d')

print('cwd:', os.getcwd())
print('py files:', [x.name for x in PROJECT_DIR.glob('*.py')])
print('end_date:', END_DATE)


In [ ]:
from r1000_data_collector import collector_lean_full_run_cfg, run_data_collection

cfg = collector_lean_full_run_cfg(BASE_DIR, end_date=END_DATE)
cfg['sec_user_agent'] = 'R1000InstitutionalBot (contact: your_email@domain.com)'  # CHANGE THIS
cfg['macro_refresh_days'] = 0
cfg['live_refresh_days'] = 0
cfg['companyfacts_refresh_days'] = 7
cfg['alpha_vantage_free_statement_refresh_days'] = 7
cfg['macro_slow_release_lag_months'] = 1
cfg['cash_target_growth_cap'] = 0.03
cfg['cash_target_balanced_cap'] = 0.05
cfg['cash_target_mild_risk_cap'] = 0.10
cfg['core_compounder_sleeve_base_weight'] = 0.80
cfg['future_winner_sleeve_base_weight'] = 0.15
cfg['future_winner_sleeve_min_weight'] = 0.05
cfg['future_winner_sleeve_max_weight'] = 0.30

collector_summary = run_data_collection(cfg)
print(collector_summary['output_files'])
print(collector_summary['core_latest_coverage'])


In [ ]:
from r1000_top30_institutional import run_default_pipeline
from r1000_data_collector import run_full_validation_suite

pipeline_cfg = dict(cfg)
pipeline_cfg['reuse_existing_artifacts'] = True
pipeline_cfg['resume_partial_walkforward'] = False
pipeline_cfg['reuse_phase4_models_for_latest_recommendations'] = False

result = run_default_pipeline(pipeline_cfg)
report = run_full_validation_suite(pipeline_cfg, rerun_pipeline=False)

print(result['acceptance_checks'])
print(report['backtest_policy_snapshot'])
print(report['rebalance_interval_comparison_snapshot'])
print(report['sleeve_policy_snapshot'])
print(report['ops_tracking_snapshot'])
print(report['coverage']['macro_scored_latest'])
print(report['portfolio_shape'])


In [ ]:
import pandas as pd

OUT = PROJECT_DIR / 'outputs'
REP = OUT / 'reports'

weights = json.loads((OUT / 'weights_latest.json').read_text(encoding='utf-8'))
run_summary = json.loads((OUT / 'run_summary.json').read_text(encoding='utf-8'))
portfolio = pd.read_csv(OUT / 'portfolio_latest.csv')
top30 = pd.read_csv(OUT / 'top30_latest.csv')
rebalance = pd.read_csv(REP / 'rebalance_interval_comparison.csv') if (REP / 'rebalance_interval_comparison.csv').exists() else pd.DataFrame()

print('weights sleeve targets:', weights.get('sleeve_target_weights'))
print('weights sleeve actual:', weights.get('sleeve_actual_weights'))
print('run_summary sleeve targets:', run_summary.get('portfolio_sleeve_target_weights'))
print('run_summary sleeve actual:', run_summary.get('portfolio_sleeve_actual_weights'))
print('run_summary rebalance action:', run_summary.get('rebalance_action'))
print('run_summary active interval:', run_summary.get('active_rebalance_interval_months'))

display(portfolio[[
    'ticker',
    'weight',
    'portfolio_sleeve_label',
    'portfolio_sleeve_confidence',
    'sleeve_target_core_compounder_weight',
    'sleeve_target_future_winner_weight',
    'future_winner_regime_strength',
    'rebalance_action',
    'active_rebalance_interval_months',
]].head(20))

display(top30[[
    'rank',
    'ticker',
    'score',
    'portfolio_sleeve_label',
    'portfolio_core_compounder_engine_score',
    'portfolio_future_winner_engine_score',
]].head(30))

display(rebalance)


In [ ]:
scored = pd.read_csv(OUT / 'scored_latest.csv')

macro_cols = [
    'm2_yoy_lag1m',
    'fed_assets_bil',
    'reverse_repo_bil',
    'tga_bil',
    'net_liquidity_bil',
    'net_liquidity_change_1m_bil',
    'liquidity_impulse_score',
    'liquidity_drain_score',
]

display(scored[[c for c in macro_cols if c in scored.columns]].notna().mean().sort_values(ascending=False))
